In [1]:
import pandas as pd

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv("cleaned_event_data.csv")

# ==========================================
# Data-Driven Severity Weights
# Derived from High Priority %
# ==========================================

severity_weights = {
    "Debris": 67,
    "Fog / Low Visibility": 50,
    "accident": 46,
    "congestion": 69,
    "construction": 63,
    "debris": 100,
    "others": 60,
    "pot_holes": 56,
    "procession": 32,
    "protest": 40,
    "public_event": 50,
    "road_conditions": 55,
    "test_demo": 33,
    "tree_fall": 33,
    "vehicle_breakdown": 66,
    "vip_movement": 35,
    "water_logging": 59
}

# ==========================================
# TII Calculation
# ==========================================

def calculate_tii(row):

    score = severity_weights.get(
        row["event_cause"],
        50
    )

    # Road Closure Bonus
    if row["requires_road_closure"]:
        score += 20

    return min(score, 100)

# ==========================================
# Create TII
# ==========================================

df["impact_score"] = df.apply(
    calculate_tii,
    axis=1
)

# ==========================================
# Check Distribution
# ==========================================

print("\nTraffic Impact Index Summary\n")

print(
    df["impact_score"].describe()
)

# ==========================================
# Risk Category
# ==========================================

def get_risk(score):

    if score >= 80:
        return "HIGH"

    elif score >= 60:
        return "MEDIUM"

    else:
        return "LOW"

df["risk_level"] = df["impact_score"].apply(
    get_risk
)

print("\nRisk Distribution\n")

print(
    df["risk_level"].value_counts()
)

# ==========================================
# Save Updated Dataset
# ==========================================

df.to_csv(
    "impact_dataset.csv",
    index=False
)

print("\nSaved: impact_dataset.csv")


Traffic Impact Index Summary

count    8173.000000
mean       63.141197
std         8.823243
min        32.000000
25%        60.000000
50%        66.000000
75%        66.000000
max       100.000000
Name: impact_score, dtype: float64

Risk Distribution

risk_level
MEDIUM    5892
LOW       1881
HIGH       400
Name: count, dtype: int64

Saved: impact_dataset.csv


In [3]:
import pandas as pd
import numpy as np

# ==========================================
# Load Dataset
# ==========================================

df = pd.read_csv("cleaned_event_data.csv")

print("Dataset Loaded")
print(df.shape)

# ==========================================
# BASIC CLEANING
# ==========================================

# Normalize common text columns
text_cols = ["event_cause", "priority", "event_type", "junction"]

for col in text_cols:
    df[col] = df[col].astype(str).str.strip()

# Lowercase important categorical fields
df["event_cause"] = df["event_cause"].str.lower()
df["priority"] = df["priority"].str.lower()
df["event_type"] = df["event_type"].str.lower()

# Fill missing junctions
df["junction"] = df["junction"].replace("nan", np.nan)
df["junction"] = df["junction"].fillna("unknown")

# ==========================================
# DUPLICATE / INCONSISTENT EVENT CAUSE FIXING
# ==========================================
# This dictionary merges same-meaning categories
# into one standard event_cause label.

event_cause_mapping = {
    "debris": "debris",
    "debris ": "debris",
    "debris/garbage": "debris",
    "fog / low visibility": "fog_low_visibility",
    "fog/low visibility": "fog_low_visibility",
    "fog low visibility": "fog_low_visibility",
    "public event": "public_event",
    "public_event": "public_event",
    "road conditions": "road_conditions",
    "road_conditions": "road_conditions",
    "test demo": "test_demo",
    "test_demo": "test_demo",
    "vip movement": "vip_movement",
    "vip_movement": "vip_movement",
    "vehicle breakdown": "vehicle_breakdown",
    "vehicle_breakdown": "vehicle_breakdown",
    "tree fall": "tree_fall",
    "tree_fall": "tree_fall",
    "water logging": "water_logging",
    "water_logging": "water_logging",
    "pot holes": "pot_holes",
    "pot_holes": "pot_holes"
}

# Apply mapping only where possible
df["event_cause"] = df["event_cause"].replace(event_cause_mapping)

# Optional: see cleaned categories
print("\nUnique Event Causes After Cleaning:\n")
print(sorted(df["event_cause"].dropna().unique()))

# ==========================================
# 1) DATA-DRIVEN EVENT CAUSE SEVERITY SCORE
# ------------------------------------------
# Based on % of HIGH priority incidents
# for each event cause
# ==========================================

df["is_high_priority"] = (df["priority"] == "high").astype(int)

cause_stats = (
    df.groupby("event_cause")["is_high_priority"]
    .mean()
    .reset_index()
)

cause_stats["cause_severity_score"] = (
    cause_stats["is_high_priority"] * 100
).round(2)

cause_severity_map = dict(
    zip(
        cause_stats["event_cause"],
        cause_stats["cause_severity_score"]
    )
)

print("\nCause Severity Weights Derived from Dataset:\n")
print(
    cause_stats.sort_values(
        "cause_severity_score",
        ascending=False
    )
)

# ==========================================
# 2) PRIORITY SCORE
# ==========================================

def get_priority_score(priority):
    if priority == "high":
        return 100
    else:
        return 50

# ==========================================
# 3) ROAD CLOSURE SCORE
# ==========================================

def get_closure_score(requires_road_closure):
    return 100 if requires_road_closure else 0

# ==========================================
# 4) EVENT TYPE SCORE
# ==========================================

def get_event_type_score(event_type):
    if event_type == "unplanned":
        return 100
    else:
        return 70

# ==========================================
# 5) HOTSPOT PRESSURE SCORE
# ------------------------------------------
# Based on frequency of historical incidents
# at a junction
# ==========================================

junction_counts = (
    df["junction"]
    .value_counts()
    .reset_index()
)

junction_counts.columns = ["junction", "event_count"]

min_count = junction_counts["event_count"].min()
max_count = junction_counts["event_count"].max()

if max_count == min_count:
    junction_counts["hotspot_score"] = 50
else:
    junction_counts["hotspot_score"] = (
        (junction_counts["event_count"] - min_count)
        / (max_count - min_count)
    ) * 100

junction_hotspot_map = dict(
    zip(
        junction_counts["junction"],
        junction_counts["hotspot_score"]
    )
)

# ==========================================
# 6) FINAL TII CALCULATION
# ==========================================
# Hybrid Formula:
#
# TII =
# 0.55 * Cause Severity
# + 0.15 * Priority Score
# + 0.15 * Road Closure Score
# + 0.10 * Event Type Score
# + 0.05 * Hotspot Pressure
#
# Final score clipped to 0-100
# ==========================================

def calculate_tii(row):

    cause_score = cause_severity_map.get(
        row["event_cause"],
        50
    )

    priority_score = get_priority_score(
        row["priority"]
    )

    closure_score = get_closure_score(
        row["requires_road_closure"]
    )

    event_type_score = get_event_type_score(
        row["event_type"]
    )

    hotspot_score = junction_hotspot_map.get(
        row["junction"],
        50
    )

    tii = (
        0.55 * cause_score
        + 0.15 * priority_score
        + 0.15 * closure_score
        + 0.10 * event_type_score
        + 0.05 * hotspot_score
    )

    return round(min(tii, 100), 2)

# ==========================================
# Create Impact Score
# ==========================================

df["impact_score"] = df.apply(
    calculate_tii,
    axis=1
)

# ==========================================
# Risk Category
# ==========================================

def get_risk(score):
    if score >= 80:
        return "HIGH"
    elif score >= 60:
        return "MEDIUM"
    else:
        return "LOW"

df["risk_level"] = df["impact_score"].apply(get_risk)

# ==========================================
# OUTPUT CHECKS
# ==========================================

print("\nTraffic Impact Index Summary\n")
print(df["impact_score"].describe())

print("\nRisk Distribution\n")
print(df["risk_level"].value_counts())

print("\nFinal Event Cause Counts\n")
print(df["event_cause"].value_counts())

# ==========================================
# Save Reference Tables
# ==========================================

cause_stats.to_csv(
    "event_cause_severity_weights.csv",
    index=False
)

junction_counts.to_csv(
    "junction_hotspot_scores.csv",
    index=False
)

# ==========================================
# Save Final Dataset
# ==========================================

df.to_csv(
    "impact_dataset.csv",
    index=False
)

print("\nSaved Files:")
print("1. impact_dataset.csv")
print("2. event_cause_severity_weights.csv")
print("3. junction_hotspot_scores.csv")

Dataset Loaded
(8173, 51)

Unique Event Causes After Cleaning:

['accident', 'congestion', 'construction', 'debris', 'fog_low_visibility', 'others', 'pot_holes', 'procession', 'protest', 'public_event', 'road_conditions', 'test_demo', 'tree_fall', 'vehicle_breakdown', 'vip_movement', 'water_logging']

Cause Severity Weights Derived from Dataset:

           event_cause  is_high_priority  cause_severity_score
3               debris          0.692308                 69.23
1           congestion          0.691176                 69.12
13   vehicle_breakdown          0.661969                 66.20
2         construction          0.629167                 62.92
5               others          0.595611                 59.56
15       water_logging          0.591703                 59.17
6            pot_holes          0.556797                 55.68
10     road_conditions          0.547059                 54.71
4   fog_low_visibility          0.500000                 50.00
9         public_even

In [4]:
import pandas as pd

# ==========================================
# Load Final Impact Dataset
# ==========================================

df = pd.read_csv("impact_dataset.csv")

print("Dataset Loaded")
print(df.shape)

# ==========================================
# Event Cause Counts After Cleaning
# ==========================================

print("\nEvent Cause Counts After Cleaning:\n")
print(df["event_cause"].value_counts())

# ==========================================
# Optional: Save Counts to CSV
# ==========================================

event_counts = (
    df["event_cause"]
    .value_counts()
    .reset_index()
)

event_counts.columns = ["event_cause", "count"]

event_counts.to_csv(
    "cleaned_event_cause_counts.csv",
    index=False
)

print("\nSaved: cleaned_event_cause_counts.csv")

Dataset Loaded
(8173, 54)

Event Cause Counts After Cleaning:

event_cause
vehicle_breakdown     4896
others                 638
pot_holes              537
construction           480
water_logging          458
accident               365
tree_fall              284
road_conditions        170
congestion             136
public_event            84
procession              72
vip_movement            20
protest                 15
debris                  13
test_demo                3
fog_low_visibility       2
Name: count, dtype: int64

Saved: cleaned_event_cause_counts.csv
